In [1]:
import os 
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client, Client

# Load environment variables from .env file
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_SERVICE_KEY = os.getenv("SUPABASE_SERVICE_KEY")

# Initialize Supabase client
supabase: Client = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)
print("Supabase client initialized successfully.")

Supabase client initialized successfully.


In [4]:
# ── Fetch live_sessions joined with period metadata ──────────────
# We select only the columns we need to keep the payload minimal.
# revenue_shopee + revenue_tiktok = total session revenue.
# period_id is the time index we will use for forecasting.

response = (
    supabase.table("live_sessions")
    .select(
        "id, date, period_id, revenue_shopee, revenue_tiktok, "
        "periods(period_id, period_name, period_start_date, period_end_date)"
    )
    .order("date", desc=False)  # Oldest → newest for time-series ordering
    .execute()
)

raw_df = pd.DataFrame(response.data)
raw_df.head()




,id,date,period_id,revenue_shopee,revenue_tiktok,periods
0,1130,2024-03-05,1,0.0,2400000.0,"{'period_id': 1, 'period_name': 'Period 1', 'p..."
1,2571,2024-03-05,1,0.0,2400000.0,"{'period_id': 1, 'period_name': 'Period 1', 'p..."
2,2572,2024-03-05,1,0.0,NaN,"{'period_id': 1, 'period_name': 'Period 1', 'p..."
3,1131,2024-03-06,1,3644300.0,507800.0,"{'period_id': 1, 'period_name': 'Period 1', 'p..."
4,2573,2024-03-06,1,3644300.0,507800.0,"{'period_id': 1, 'period_name': 'Period 1', 'p..."


In [5]:
# ── Flatten the nested period object returned by Supabase ────────
# Supabase returns related rows as nested dicts: {"periods": {"period_name": ...}}
# We normalise this so each period field becomes a top-level column.

period_meta = pd.json_normalize(raw_df["periods"]).add_prefix("period_")
raw_df = pd.concat([raw_df.drop(columns=["periods"]), period_meta], axis=1)

# ── Compute total revenue per session ─────────────────────────
# Replace None with 0 before summing (some sessions may be single-platform)
raw_df["revenue_shopee"]  = raw_df["revenue_shopee"].fillna(0).astype(int)
raw_df["revenue_tiktok"]  = raw_df["revenue_tiktok"].fillna(0).astype(int)
raw_df["revenue_total"]   = raw_df["revenue_shopee"] + raw_df["revenue_tiktok"]

# ── Aggregate: total revenue per period ───────────────────────
# Each period represents one billing/performance cycle (from the periods table).
# This is the "Y" variable (what we are forecasting).
period_revenue_df = (
    raw_df
    .groupby(["period_id", "period_period_name", "period_period_start_date"], as_index=False)
    .agg(
        actual_revenue   = ("revenue_total",   "sum"),
        session_count    = ("id",              "count"),
        avg_shopee_rev   = ("revenue_shopee",  "mean"),
        avg_tiktok_rev   = ("revenue_tiktok",  "mean"),
    )
    .rename(columns={
        "period_period_name":       "period_name",
        "period_period_start_date": "period_start_date",
    })
    .sort_values("period_id")
    .reset_index(drop=True)
)

print(f"✓ Aggregated into {len(period_revenue_df)} period revenue rows")
period_revenue_df


✓ Aggregated into 7 period revenue rows


,period_id,period_name,period_start_date,actual_revenue,session_count,avg_shopee_rev,avg_tiktok_rev
0,1,Period 1,2025-03-03,393286038,86,3.907898e+06,6.651958e+05
1,2,Period 2,2025-04-05,622309567,147,3.964171e+06,2.692272e+05
2,3,Period 3,2025-05-02,653124767,200,2.968658e+06,2.969655e+05
3,4,Period 4,2025-08-30,531823266,171,2.799366e+06,3.107115e+05
4,5,Period 5,2025-10-01,419628196,180,2.250813e+06,8.045506e+04
5,6,Period 6,2025-10-31,170490846,110,1.481904e+06,6.801278e+04
6,7,Period 7,2025-03-02,801683555,106,4.986714e+06,2.576338e+06


In [6]:
# ── Save extracted data for the next step ────────────────────────
# We export to a local CSV so each notebook can run independently.
# Defence: this makes the pipeline re-runnable without re-fetching the DB.

period_revenue_df.to_csv("extracted_period_revenue.csv", index=False)
print("✓ Saved: extracted_period_revenue.csv")


✓ Saved: extracted_period_revenue.csv
